In [ ]:
import sys
import os
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt

from sklearn.decomposition import PCA


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_fly import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
dims = 3

# read_ids = list(range(0, 12+1))
read_ids = [0,   # paper
			2,
			3,
			4,
			5,
			6,
			8,
			9]   # good
# read_ids = [0, 2, 3, 8, 9]
# read_ids = [6]
plot_id = 0

In [ ]:
# read

path = os.path.join(read_utils.get_data_path(), 'brain')
(file_ids, filenames,
 traces,
 states, markers) = read_utils.get_data(path, read_ids)

# # clear nan values
# for i in range(len(file_ids)):
# 	mask = markers[i].notna().all(axis=1)
#
# 	traces[i] = traces[i][mask]
# 	markers[i] = markers[i][mask]
# 	states[i] = states[i][mask]
#
# 	#filenames[i] = filenames[i][mask]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

In [ ]:
# prep

def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
    return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

sampling_rates = [read_utils.get_sampling_rate(os.path.join(path, filenames[file_ids.index(file_id)])) for file_id in file_ids]

traces = [pd.DataFrame(highpass_butter(trace.values, sr, 1 / 100), columns=trace.columns) for trace, sr in zip(traces, sampling_rates)]

markers_grad = [pd.DataFrame(np.gradient(marker.values, axis=0), columns=marker.columns) for marker in markers]

In [ ]:
selection_bhl = ['walk', 'still', 'front-groom', 'back-groom', 'abdomen-move', 'background']

plot_idx = file_ids.index(plot_id)

# discrete_states = [ds[selection_bhl] for ds in discrete_states]

In [ ]:
(modes_manifold,
 recos_manifold,
 loads_pca_man,
 vars_expl_mans) = train_utils.get_manifolds_pca(traces, zscore=False,
												 n_pc=dims)

In [ ]:
save = False

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_manifold, states,
						  [vx[:2] for vx in vars_expl_mans], None,
						  title='PCA manifolds (100s high-pass filter)', top_space=2.5,
						  save_path=f'../../out/plots/manifolds/{dims}-dim-PCA' if save else None)

In [ ]:
# plot_utils.plot_manifolds_3d([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 							 modes_manifold, states,
# 							 [vx[:3] for vx in vars_expl_mans], None,
# 							 title='PCA manifolds (100s high-pass filter)', top_space=3,
# 							 save_path=f'../../out/plots/manifolds/{dims}-dim-PCA_3d' if save else None)

In [ ]:
traces_manifolds_indices = [np.argsort(xloads_pca_man[0, :]) for xloads_pca_man in loads_pca_man]

traces_sorted = [t.iloc[:, t_man_idx.tolist()] for t, t_man_idx in zip(traces, traces_manifolds_indices)]

traces_manifolds_sorted = [t_man.iloc[:, t_man_idx.tolist()] for t_man, t_man_idx in zip(recos_manifold, traces_manifolds_indices)]

traces_residuals = [pd.DataFrame(data=t.values - t_man.values, columns=t.columns) for t, t_man in zip(traces, recos_manifold)]

traces_residuals_sorted_by_manifold = [t_res.iloc[:, t_man_idx.tolist()] for t_res, t_man_idx in zip(traces_residuals, traces_manifolds_indices)]

# fly 0

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_sorted[file_ids.index(0)],
	None,
	'traces', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_manifolds_sorted[file_ids.index(0)],
	None,
	'manifold', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_residuals_sorted_by_manifold[file_ids.index(0)],
	None,
	'residuals', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

# fly 2

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_sorted[file_ids.index(2)],
	None,
	'traces', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_manifolds_sorted[file_ids.index(2)],
	None,
	'manifold', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_residuals_sorted_by_manifold[file_ids.index(2)],
	None,
	'residuals', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

# fly 3

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_sorted[file_ids.index(3)],
	None,
	'traces', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_manifolds_sorted[file_ids.index(3)],
	None,
	'manifold', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_residuals_sorted_by_manifold[file_ids.index(3)],
	None,
	'residuals', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

# fly 8


In [ ]:
plot_utils.plot_heatmap_stack(
	traces_sorted[file_ids.index(8)],
	None,
	'traces', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_manifolds_sorted[file_ids.index(8)],
	None,
	'manifold', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_residuals_sorted_by_manifold[file_ids.index(8)],
	None,
	'residuals', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

# fly 9

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_sorted[file_ids.index(9)],
	None,
	'traces', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_manifolds_sorted[file_ids.index(9)],
	None,
	'manifold', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)

In [ ]:
plot_utils.plot_heatmap_stack(
	traces_residuals_sorted_by_manifold[file_ids.index(9)],
	None,
	'residuals', None,
	zlimits_activity=(-0.015, 0.015), cmap='plasma', neuron_ids=True)